# Carga de imagenes al datastore y mosaic dataset

Flujo operativo posterior a la preparacion del notebook `008`. Usa `04_ready_for_datastore.csv` para copiar archivos al datastore y `06_attribute_updates.csv` para actualizar atributos del mosaic dataset.

In [1]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_loader as mosaic_loader
mosaic_loader = importlib.reload(mosaic_loader)
from core.mosaic_loader import *

# PARAMETROS
RUN_PREPARACION = "20260615_155625"
OUTPUT_PREPARACION_DIR = Path.cwd() / "outputs" / "preparacion_carga_mosaico" / RUN_PREPARACION

READY_FOR_DATASTORE_CSV = OUTPUT_PREPARACION_DIR / "04_ready_for_datastore.csv"
ATTRIBUTE_UPDATES_CSV = OUTPUT_PREPARACION_DIR / "06_attribute_updates.csv"

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

# Seguridad operacional: validar primero con DRY_RUN=True. Cambiar a False para ejecutar copia/carga/update.
DRY_RUN = False
OVERWRITE_COPY = False
SKIP_EXISTING_MOSAIC_NAME = True

# Valores fijos definidos para esta carga.
MAXPS_VALUE = 10000
LOWPS_VALUE = 0.15

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs" / "carga_mosaico" / run_timestamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Preparacion:", OUTPUT_PREPARACION_DIR)
print("CSV carga:", READY_FOR_DATASTORE_CSV)
print("CSV atributos:", ATTRIBUTE_UPDATES_CSV)
print("Mosaic dataset:", PATH_MOSAIC_DATASET)
print("Salida:", OUTPUT_DIR)
print("DRY_RUN:", DRY_RUN)

Preparacion: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_155625
CSV carga: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_155625\04_ready_for_datastore.csv
CSV atributos: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_155625\06_attribute_updates.csv
Mosaic dataset: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport
Salida: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\carga_mosaico\20260615_162625
DRY_RUN: False


## 1. Cargar manifiestos

Se valida que cada imagen lista tenga atributos asociados antes de ejecutar cualquier operacion.

In [2]:
load_df = load_ready_and_attributes(READY_FOR_DATASTORE_CSV, ATTRIBUTE_UPDATES_CSV)

required_columns = ["path", "destination_path", "Name", "Sector", "Fecha_Adqui", "URL", "Proyecto", "Sensor", "Fecha_Publ"]
missing_columns = [column for column in required_columns if column not in load_df.columns]
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

validation_summary = pd.DataFrame(
    [
        {"metric": "records_to_process", "value": len(load_df)},
        {"metric": "missing_source_path", "value": int((~load_df["path"].map(lambda value: Path(value).exists())).sum())},
        {"metric": "missing_destination_path", "value": int(load_df["destination_path"].isna().sum())},
        {"metric": "unique_destination_paths", "value": int(load_df["destination_path"].nunique())},
        {"metric": "unique_names", "value": int(load_df["Name"].nunique())},
    ]
)

display(validation_summary)
display(load_df[["file_name", "Name", "destination_path", "Sector", "Fecha_Adqui", "Proyecto", "Sensor", "Fecha_Publ"]].head(20))

,metric,value
0,records_to_process,32
1,missing_source_path,0
2,missing_destination_path,0
3,unique_destination_paths,32
4,unique_names,32


,file_name,Name,destination_path,Sector,Fecha_Adqui,Proyecto,Sensor,Fecha_Publ
0,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_D...,Estacion_Cabecera,2026-05-01,PAO,DJI Mavic Enterprise,2026-06-15
1,GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,Subestacion-El-Mauro,2026-05-06,PAO,DJI Mavic Enterprise,2026-06-15
2,GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,Subestacion-El-Mauro,2026-05-06,PAO,DJI Mavic Enterprise,2026-06-15
3,GEOSP-TRN-002555_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,TORRE_E85_A_E_125,2026-05-06,PAO,DJI Mavic Enterprise,2026-06-15
4,GEOSP-TRN-002591_GS_Ortofoto ED1_10-05-2026.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ED1,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
5,GEOSP-TRN-002592_GS_Ortofoto_Helipuerto Mauro ...,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,Helipuerto,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
6,GEOSP-TRN-002593_GS_ORTOFOTO_PATIO 19B_10-05-2...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,Patio-19B-y-Armado,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
7,GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,DME9-PA12-IIFF8,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
8,GEOSP-TRN-002604_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,TORRES_E48_A_E84_PV4,2026-05-07,PAO,DJI Mavic Enterprise,2026-06-15
9,GEOSP-TRN-002606_GS_Ortofoto_Tramo 1 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15


## 2. Ejecutar copia, carga al mosaico, footprints y atributos

Con `DRY_RUN=True` no escribe archivos ni modifica el mosaic dataset. Con `DRY_RUN=False` ejecuta el flujo completo por imagen.

In [3]:
results = []

for index, row in load_df.iterrows():
    print(f"[{index + 1}/{len(load_df)}] {row['Name']}")
    result = process_mosaic_load_row(
        row,
        PATH_MOSAIC_DATASET,
        overwrite_copy=OVERWRITE_COPY,
        skip_existing_mosaic_name=SKIP_EXISTING_MOSAIC_NAME,
        maxps_value=MAXPS_VALUE,
        lowps_value=LOWPS_VALUE,
        dry_run=DRY_RUN,
    )
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df)
display(results_df["overall_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "overall_status"}))

[1/32] CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera
[2/32] CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-1
[3/32] CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-2
[4/32] CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125
[5/32] CL_MLP_PAO_IF_Ortho_26_05_10_ED1
[6/32] CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto
[7/32] CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado
[8/32] CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8
[9/32] CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4
[10/32] CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3
[11/32] CL_MLP_PAO_IF_Ortho_26_05_13_DME9-PA12-IIFF8
[12/32] CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro_A_E35
[13/32] CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-1
[14/32] CL_MLP_PAO_IF_Ortho_26_05_13_EM2_S2
[15/32] CL_MLP_PAO_IF_Ortho_26_05_13_EBD-1
[16/32] CL_MLP_PAO_IF_Ortho_26_05_13_ED2
[17/32] CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-2
[18/32] CL_MLP_PAO_IF_Ortho_26_05_13_EBD-2
[19/32] CL_MLP_PAO_IF_Ortho_26_05_13_Estacion_

,file_name,Name,destination_path,overall_status,source_path,copy_status,copy_error,mosaic_add_status,mosaic_add_error,footprint_status,footprint_error,attribute_status,attribute_rows_updated,attribute_missing_fields,attribute_error
0,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_D...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
1,GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
2,GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
3,GEOSP-TRN-002555_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
4,GEOSP-TRN-002591_GS_Ortofoto ED1_10-05-2026.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
5,GEOSP-TRN-002592_GS_Ortofoto_Helipuerto Mauro ...,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
6,GEOSP-TRN-002593_GS_ORTOFOTO_PATIO 19B_10-05-2...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
7,GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
8,GEOSP-TRN-002604_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
9,GEOSP-TRN-002606_GS_Ortofoto_Tramo 1 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None


,overall_status,count
0,ok,32


## 3. Exportar resultados de ejecucion

In [4]:
summary_rows = [
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "preparation_run", "value": RUN_PREPARACION},
    {"metric": "dry_run", "value": DRY_RUN},
    {"metric": "mosaic_dataset", "value": PATH_MOSAIC_DATASET},
    {"metric": "records_to_process", "value": len(load_df)},
    {"metric": "maxps_value", "value": MAXPS_VALUE},
    {"metric": "lowps_value", "value": LOWPS_VALUE},
]

for column in ["copy_status", "mosaic_add_status", "footprint_status", "attribute_status", "overall_status"]:
    if column in results_df.columns:
        for status, count in results_df[column].value_counts(dropna=False).items():
            summary_rows.append({"metric": f"{column}_{status}", "value": int(count)})

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / "00_summary.csv"
results_csv = OUTPUT_DIR / "01_load_results.csv"
errors_csv = OUTPUT_DIR / "02_errors.csv"

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
results_df.to_csv(results_csv, index=False, encoding="utf-8-sig")
error_columns = [column for column in results_df.columns if column.endswith("_error")]
error_filter = results_df[error_columns].notna().any(axis=1) if error_columns else pd.Series(False, index=results_df.index)
results_df[error_filter].to_csv(errors_csv, index=False, encoding="utf-8-sig")

display(summary_df)
print("Resultados exportados en:", OUTPUT_DIR)

,metric,value
0,run_timestamp,20260615_162625
1,preparation_run,20260615_155625
2,dry_run,False
3,mosaic_dataset,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
4,records_to_process,32
5,maxps_value,10000
6,lowps_value,0.15
7,copy_status_copied,32
8,mosaic_add_status_added,32
9,footprint_status_built,32


Resultados exportados en: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\carga_mosaico\20260615_162625


In [5]:
Name = 'CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera'
load_df['Name'].tolist()

['CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera',
 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-1',
 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-2',
 'CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125',
 'CL_MLP_PAO_IF_Ortho_26_05_10_ED1',
 'CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto',
 'CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado',
 'CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8',
 'CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4',
 'CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3',
 'CL_MLP_PAO_IF_Ortho_26_05_13_DME9-PA12-IIFF8',
 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro_A_E35',
 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-1',
 'CL_MLP_PAO_IF_Ortho_26_05_13_EM2_S2',
 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-1',
 'CL_MLP_PAO_IF_Ortho_26_05_13_ED2',
 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-2',
 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-2',
 'CL_MLP_PAO_IF_Ortho_26_05_13_Estacion_Intermedia',
 'CL_MLP_PAO_IF_Ortho_26_05_14_EM3',
 'CL_MLP_PAO_IF_Ort

In [7]:
q = tuple(results_df['Name'])
q = f'Name IN {q}'
q

"Name IN ('CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera', 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-1', 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-2', 'CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125', 'CL_MLP_PAO_IF_Ortho_26_05_10_ED1', 'CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto', 'CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado', 'CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8', 'CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4', 'CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_26_05_13_DME9-PA12-IIFF8', 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro_A_E35', 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-1', 'CL_MLP_PAO_IF_Ortho_26_05_13_EM2_S2', 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-1', 'CL_MLP_PAO_IF_Ortho_26_05_13_ED2', 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-2', 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-2', 'CL_MLP_PAO_IF_Ortho_26_05_13_Estacion_Intermedia', 'CL_MLP_PAO_IF_Ortho_26_05_14_EM3', 'CL_MLP_PAO_IF_Ortho_26_05_14

In [ ]:
import arcpy

Estado = 'Activo'
with arcpy.da.SearchCursor(
    PATH_MOSAIC_DATASET,
    ['NombreVuelo'],
    where_clause=None,
    sql_clause=(None, "ORDER BY Name ASC")
) as cursor:
    for c in cursor:
        if c[0] is not None:
            print(c)
            break

('143_25_12_19_Plataforma_Integrada_El_Mauro',)


In [ ]:
# with arcpy.da.UpdateCursor(
#     PATH_MOSAIC_DATASET,
#     ['Estado'],
#     where_clause=q
    
# ) as cursor:
#     for c in cursor:
#         c[0] = 'Activo'
#         cursor.updateRow(c)

['Activo']

## Se cargan los footprint

In [54]:
arcpy.env.overwriteOutput = True
fc_footprint_temp = 'in_memory/foot3'
arcpy.ExportMosaicDatasetGeometry_management(PATH_MOSAIC_DATASET,
                                             out_feature_class=fc_footprint_temp,
                                             where_clause=q)


<Result 'in_memory\\foot3'>

In [55]:
arcpy.GetCount_management(fc_footprint_temp)[0]

'32'

In [56]:
import re
fc_footprint = r'\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO'

mapsfields = {
    'FechaAdqui':'Fecha_Adqu', 
    'FechaCarga':'Fecha_Publ' ,
    'NombreVuelo':'Nombre_de_Vuelo', 
    'ProductName':'ProductNam'
    }
## Se renombran los campos de la exportacion
for f in mapsfields:
    old_name = f
    new_name = mapsfields[f]
    print(f'{old_name} ==> {new_name}')
    arcpy.AlterField_management(fc_footprint_temp,old_name,new_name)

FechaAdqui ==> Fecha_Adqu
FechaCarga ==> Fecha_Publ
NombreVuelo ==> Nombre_de_Vuelo
ProductName ==> ProductNam


In [57]:
arcpy.Append_management(fc_footprint_temp,fc_footprint,'NO_TEST','')

<Result '\\\\amssclgis08.ams.gmams.cl\\CL_MLP_PAO\\02_FGDB\\CL_MLP_PAO_v1.gdb\\CL_MLP_PAO_06_COMPLEMENTOS\\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO'>

In [58]:
len([c for c in arcpy.da.SearchCursor(fc_footprint,['Nombre_de_Vuelo'],"Nombre_de_Vuelo is null")])

36

In [73]:
cols = ['Nombre_de_Vuelo','Sector','Fecha_Adqu']
with arcpy.da.UpdateCursor(
    fc_footprint,
    cols,
    where_clause=None,
    sql_clause=(None, "ORDER BY Name ASC")
) as cursor:
    cnt = 0
    for c in cursor:
        if c[1] and c[2]:
            cnt +=1
            fecha = pd.to_datetime(c[2]).strftime('%y_%m_%d')
            numero = str(cnt).zfill(3)
            sector = c[1]
            name_ = f'{numero}_{fecha}_{sector}'
            c[0] = name_
            cursor.updateRow(c)
            print(name_)
       

001_25_01_08_EB1
002_25_02_26_RutaD835
003_25_03_10_NSTC_116a118
004_25_03_10_NSTC_118a120
005_25_03_12_SRA2
006_25_03_17_NSTC_Km_120_a_121
007_25_03_19_Orejas16_Ruta-D-865
008_25_03_19_Orejas17_Ruta-D-865
009_25_03_19_Orejas18
010_25_03_31_EB2
011_25_03_31_EV2
012_25_04_02_DME-13
013_25_04_02_Patio-Acopio-17
014_25_04_02_Ruta-SE-a-DME-13
015_25_04_03_Area-patio-19b-y-armado
016_25_04_03_Subestacion-El-Mauro
017_25_04_09_Cachimba_de_Bajo_Camisas_ED2
018_25_04_10_Sector_Pupio_I_Area_1
019_25_05_07_Monte-Aranda-84p2-a-82p3
020_25_05_07_Monte_Aranda_82p3_a_80p7
021_25_05_12_EDT
022_25_05_12_IIFF15-Campamento-Tipay
023_25_05_15_DME9_PA12_IIFF8
024_25_05_15_ED1_IIFF7_DME8
025_25_05_15_EM2_PA11_01
026_25_05_29_DME5A_DME17_a_NSTC_78p9
027_25_06_02_ByPassDrenes_Estacion_Drenes_El_Mauro
028_25_06_02_Estacion_Drenes_El_Mauro
029_25_06_04_CaminoAcceso03_LasAnimas
030_25_06_04_EM3
031_25_06_05_33-kv-74p7-a-75p5
032_25_06_05_33-kV-NSTC-km-75p5-a-76p4
033_25_06_09_EB2
034_25_06_09_EV2
035_25_06_09_E

In [80]:
## Se actualiza en los footprint
querymosaic = [c[0] for c in arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name'])]

In [81]:
# querymosaic = f"Name IN {tuple(querymosaic)}"
cursor = arcpy.da.SearchCursor(fc_footprint,['Name','Nombre_de_Vuelo'])
df = pd.DataFrame(cursor,columns=['Name','Nombre_de_Vuelo'])
print(f'total de registros {df.shape[0]}')
df = df[df.Name.isin(querymosaic)]
print(f'total de registros {df.shape[0]}') 
df.head()

total de registros 483
total de registros 340


,Name,Nombre_de_Vuelo
141,CL_MLP_PAO_IF_Ortho_26_01_03_DME5A-DME17-a-NST...,145_26_01_03_DME5A-DME17-a-NSTC-78p9
142,CL_MLP_PAO_IF_Ortho_26_01_10_MonteAranda-NSTC-...,156_26_01_10_MonteAranda-NSTC-Km-84p2-a-82p3
143,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,165_26_01_17_MonteAranda-NSTC-Km-84p2-a-82p3
144,CL_MLP_PAO_IF_Ortho_26_01_21_InstalacionesTipay,171_26_01_21_InstalacionesTipay
145,CL_MLP_PAO_IF_Ortho_26_01_11_EM1,157_26_01_11_EM1


In [98]:
result = []
with arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name','NombreVuelo']) as cursor:
    for c in cursor:
        try:
            name = c[0]
            nombre_vuelo = df.loc[df.Name == name,'Nombre_de_Vuelo'].values[0]
            result.append([c[1],nombre_vuelo])
        except:
            continue
        
df2 = pd.DataFrame(result,columns=['NombreVueloOld','NombreVuelo'])
df2.head()

,NombreVueloOld,NombreVuelo
0,148_26_01_07_ED2,147_26_01_07_ED2
1,379_26_04_11_Camino_Alternativo_Salamanca,364_26_04_11_Camino_Alternativo_Salamanca
2,None,469_26_05_14_Camino_Alternativo_Salamanca
3,394_26_04_16_Camino_Alternativo_Salamanca,379_26_04_16_Camino_Alternativo_Salamanca
4,342_26_04_02_Camino_Alternativo_Salamanca,329_26_04_02_Camino_Alternativo_Salamanca


In [ ]:
query_update = tuple(df2.NombreVueloOld)
query_update = f'NombreVuelo IN {query_update}'
dict_rename = dict(zip(df2.NombreVueloOld,df2.NombreVuelo))

In [109]:
with arcpy.da.UpdateCursor(PATH_MOSAIC_DATASET,['NombreVuelo']) as cursor:
    for c in cursor:
        new_name_ = dict_rename.get(c[0],None)
        if new_name_:
            c[0] = new_name
            cursor.updateRow(c)
            

'147_26_01_07_ED2'

'148_26_01_07_ED2'

In [ ]:
aprx_path = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
map_name = "CL MLP PAO 27 Imagenes Aereas PAO Image Server"
prefix = "CL_MLP_PAO_IF_Ortho_"
suffix = ".tif"
 
aprx = arcpy.mp.ArcGISProject(aprx_path)
maps = aprx.listMaps(map_name)
 
if not maps:
    print("No se encontró el mapa")
else:
    m = maps[0]
 
    # 1️⃣ Renombrar
    for lyr in m.listLayers():
        try:
            original_name = lyr.name
            new_name = original_name
            if new_name.startswith(prefix):
                new_name = new_name.replace(prefix, "", 1)
            if new_name.endswith(suffix):
                new_name = new_name[:-len(suffix)]
            if new_name != original_name:
                print(f"Renombrando: {original_name} -> {new_name}")
                lyr.name = new_name
        except Exception as e:
            print(f"Error con capa {lyr.name}: {e}")
 
#     # 2️⃣ Ordenar
#     layers = [lyr for lyr in m.listLayers() if lyr.isFeatureLayer or lyr.isRasterLayer]
#     layers_sorted = sorted(layers, key=lambda l: l.name)
#     for lyr in layers_sorted:
#         m.moveLayer(m.listLayers()[0], lyr, "BEFORE")
#     print("Capas ordenadas")
 
#     # 3️⃣ Guardar una sola vez
#     aprx.save()
#     print("Proyecto guardado")
 
# del aprx  # buena práctica siempre

In [19]:
maps

In [111]:
load_df.columns

Index(['ready_for_datastore', 'review_reason', 'path', 'relative_path',
       'file_name', 'expected_file_name', 'destination_path',
       'original_expected_file_name', 'expected_name', 'expected_date_token',
       'destination_folder', 'destination_date_folder', 'expected_sector',
       'sector_source', 'rename_status', 'spatial_status',
       'spatial_sector_raw', 'spatial_sector', 'spatial_overlap_pct',
       'spatial_overlap_count', 'spatial_all_matches',
       'duplicate_expected_file_name', 'duplicate_sequence',
       'duplicate_was_resolved', 'size_mb', 'modified_at', 'Name', 'Raster',
       'Path_Destino', 'Sector', 'Fecha_Adqui', 'URL', 'Proyecto', 'Sensor',
       'Fecha_Publ', 'relative_path_attr'],
      dtype='object')

In [114]:
load_df.loc[0,'destination_path']

'\\\\amssclgis10.ams.gmams.cl\\CL_MLP_PAO\\Chacay_Drone\\26_05\\CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera.tif'

In [119]:
from pathlib import Path
import csv
import arcpy

LOAD_RESULTS_CSV = Path.cwd() / "outputs" / "carga_mosaico" / "20260615_162625" / "01_load_results.csv"

aprx_path = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
map_name = "CL MLP PAO 27 Imagenes Aereas PAO Image Server"
TARGET_GROUP_LAYER_NAME = None  # Opcional: escribir aqui el nombre exacto del grupo si quieres forzarlo.
prefix = "CL_MLP_PAO_IF_Ortho_"
suffix = ".tif"
SAVE_COPY_FOR_REVIEW = True
ADD_ONLY_MISSING_TO_GROUP = True  # La lista del CSV se agrega como nuevas capas al grupo existente.
APRX_COPY_PATH = f"{Path.cwd()}\APRX\{Path(aprx_path).stem}_verificacion_carga_imagenes.aprx"

def clean_layer_name(name):
    clean_name = Path(str(name)).name
    if clean_name.startswith(prefix):
        clean_name = clean_name.replace(prefix, "", 1)
    if clean_name.endswith(suffix):
        clean_name = clean_name[:-len(suffix)]
    return clean_name


def read_loaded_image_paths(csv_path):
    with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = csv.DictReader(file)
        return [
            row["destination_path"]
            for row in rows
            if row.get("overall_status") == "ok" and row.get("destination_path")
        ]


def layer_long_name(layer):
    try:
        return layer.longName
    except Exception:
        return layer.name


def is_child_of_group(layer, group_layer):
    long_name = layer_long_name(layer)
    group_long_name = layer_long_name(group_layer)
    return long_name.startswith(group_long_name + "\\")


def layer_matches_image_naming(layer):
    name = Path(str(layer.name)).name
    return name.startswith(prefix) or name.endswith(suffix) or clean_layer_name(name) != name


def find_target_group_layer(map_obj, explicit_group_name=None):
    groups = [lyr for lyr in map_obj.listLayers() if lyr.isGroupLayer]

    if explicit_group_name:
        matches = [lyr for lyr in groups if lyr.name == explicit_group_name or layer_long_name(lyr) == explicit_group_name]
        if not matches:
            available = [layer_long_name(lyr) for lyr in groups]
            raise ValueError(f"No se encontro el grupo '{explicit_group_name}'. Grupos disponibles: {available}")
        return matches[0]

    candidates = []
    for group_layer in groups:
        child_layers = [lyr for lyr in map_obj.listLayers() if lyr != group_layer and is_child_of_group(lyr, group_layer)]
        image_children = [lyr for lyr in child_layers if (lyr.isRasterLayer or lyr.isFeatureLayer) and layer_matches_image_naming(lyr)]
        if image_children:
            candidates.append((len(image_children), group_layer))

    if not candidates:
        available = [layer_long_name(lyr) for lyr in groups]
        raise ValueError(
            "No se pudo detectar automaticamente el grupo destino. "
            f"Define TARGET_GROUP_LAYER_NAME. Grupos disponibles: {available}"
        )

    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1]


def group_child_layers(map_obj, group_layer):
    return [
        lyr
        for lyr in map_obj.listLayers()
        if lyr != group_layer and is_child_of_group(lyr, group_layer) and (lyr.isFeatureLayer or lyr.isRasterLayer)
    ]


def add_raster_to_group(map_obj, group_layer, image_path, target_name):
    top_layer = map_obj.addDataFromPath(image_path)
    top_layer.name = target_name

    grouped_layers = map_obj.addLayerToGroup(group_layer, top_layer, "BOTTOM")
    map_obj.removeLayer(top_layer)

    if grouped_layers:
        grouped_layer = grouped_layers[0] if isinstance(grouped_layers, list) else grouped_layers
        grouped_layer.name = target_name
        return grouped_layer

    for lyr in group_child_layers(map_obj, group_layer):
        if clean_layer_name(lyr.name).lower() == target_name.lower():
            lyr.name = target_name
            return lyr

    return None


image_paths_to_add = read_loaded_image_paths(LOAD_RESULTS_CSV)
print(f"Imagenes cargadas a revisar en APRX: {len(image_paths_to_add)}")

aprx = arcpy.mp.ArcGISProject(aprx_path)

try:
    maps = aprx.listMaps(map_name)

    if not maps:
        print("No se encontro el mapa")
    else:
        m = maps[0]
        target_group = find_target_group_layer(m, TARGET_GROUP_LAYER_NAME)
        print(f"Grupo destino: {layer_long_name(target_group)}")

        existing_group_layers_before = group_child_layers(m, target_group)
        existing_names = {clean_layer_name(lyr.name).lower() for lyr in existing_group_layers_before}
        print(f"Capas existentes en el grupo antes de agregar: {len(existing_group_layers_before)}")
        added_count = 0
        skipped_count = 0
        error_count = 0

        # 1. Agregar al grupo destino solo las imagenes nuevas del CSV.
        #    Las capas que ya existen en el grupo se conservan y no se reemplazan.
        for image_path in image_paths_to_add:
            target_name = clean_layer_name(Path(image_path).name)
            target_key = target_name.lower()

            if ADD_ONLY_MISSING_TO_GROUP and target_key in existing_names:
                print(f"Ya existe en el grupo, se conserva y se omite: {target_name}")
                skipped_count += 1
                continue

            try:
                grouped_layer = add_raster_to_group(m, target_group, image_path, target_name)
                existing_names.add(target_key)
                added_count += 1
                print(f"Agregada al grupo: {target_name}")
            except Exception as exc:
                error_count += 1
                print(f"Error agregando {image_path}: {exc}")

        # 2. Renombrar solamente las capas del grupo destino con la nomenclatura limpia.
        renamed_count = 0
        for lyr in group_child_layers(m, target_group):
            try:
                original_name = lyr.name
                new_name = clean_layer_name(original_name)
                if new_name != original_name:
                    print(f"Renombrando: {original_name} -> {new_name}")
                    lyr.name = new_name
                    renamed_count += 1
            except Exception as exc:
                print(f"Error con capa {lyr.name}: {exc}")

        # 3. Ordenar alfabeticamente solamente las capas del grupo destino.
        layers = group_child_layers(m, target_group)
        for lyr in sorted(layers, key=lambda layer: layer.name, reverse=True):
            current_group_layers = group_child_layers(m, target_group)
            first_layer = current_group_layers[0] if current_group_layers else None
            if first_layer and lyr != first_layer:
                m.moveLayer(first_layer, lyr, "BEFORE")

        print("Capas del grupo ordenadas")
        existing_group_layers_after = group_child_layers(m, target_group)
        print(f"Capas existentes en el grupo despues de agregar: {len(existing_group_layers_after)}")
        print(f"Resumen APRX: nuevas_agregadas={added_count}, existentes_conservadas={skipped_count}, renombradas={renamed_count}, errores={error_count}")

        # 4. Guardar el proyecto o una copia de revision.
        if SAVE_COPY_FOR_REVIEW:
            aprx.saveACopy(APRX_COPY_PATH)
            print(f"Copia de verificacion guardada: {APRX_COPY_PATH}")
        else:
            aprx.save()
            print("Proyecto original guardado")
finally:
    del aprx




Imagenes cargadas a revisar en APRX: 32
Grupo destino: Vuelos Drone PAO
Capas existentes en el grupo antes de agregar: 468
Agregada al grupo: 26_05_01_Estacion_Cabecera
Agregada al grupo: 26_05_06_Subestacion-El-Mauro-1
Agregada al grupo: 26_05_06_Subestacion-El-Mauro-2
Agregada al grupo: 26_05_06_TORRE_E85_A_E_125
Agregada al grupo: 26_05_10_ED1
Agregada al grupo: 26_05_10_Helipuerto
Agregada al grupo: 26_05_10_Patio-19B-y-Armado
Agregada al grupo: 26_05_10_DME9-PA12-IIFF8
Agregada al grupo: 26_05_07_TORRES_E48_A_E84_PV4
Agregada al grupo: 26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3
Agregada al grupo: 26_05_13_DME9-PA12-IIFF8
Agregada al grupo: 26_05_13_Subestacion-El-Mauro_A_E35
Agregada al grupo: 26_05_13_Subestacion-El-Mauro-1
Agregada al grupo: 26_05_13_EM2_S2
Agregada al grupo: 26_05_13_EBD-1
Agregada al grupo: 26_05_13_ED2
Agregada al grupo: 26_05_13_Subestacion-El-Mauro-2
Agregada al grupo: 26_05_13_EBD-2
Agregada al grupo: 26_05_13_Estacion_Intermedia
Agregada al grupo: 26_05_14_

In [122]:
fc_footprint
q

"Name IN ('CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera', 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-1', 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-2', 'CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125', 'CL_MLP_PAO_IF_Ortho_26_05_10_ED1', 'CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto', 'CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado', 'CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8', 'CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4', 'CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_26_05_13_DME9-PA12-IIFF8', 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro_A_E35', 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-1', 'CL_MLP_PAO_IF_Ortho_26_05_13_EM2_S2', 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-1', 'CL_MLP_PAO_IF_Ortho_26_05_13_ED2', 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-2', 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-2', 'CL_MLP_PAO_IF_Ortho_26_05_13_Estacion_Intermedia', 'CL_MLP_PAO_IF_Ortho_26_05_14_EM3', 'CL_MLP_PAO_IF_Ortho_26_05_14

In [136]:
PATH_MOSAIC_DATASET
cursor = arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name','URL','OBJECTID'],q)
df = pd.DataFrame(cursor,columns=['Name','URL','OBJECTID'])

In [139]:
with arcpy.da.UpdateCursor(fc_footprint,['Name','URL','ProductNam'],q) as cursor:
    for c in cursor:
    
        # url_ = df.loc[df.Name == c[0],'URL'].values[0]
        # url_
        # c[1] = url_
        obj = df.loc[df.Name == c[0],'OBJECTID'].values[0]
        c[2] = str(obj)
        
        cursor.updateRow(c)

In [138]:
c

['CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera',
 'https://sig.aminerals.cl/imgdyn/rest/services/CL_MLP_PAO/CL_MLP_PAO_IF_Ortho_Geosupport/ImageServer/file?id=.\\Chacay_Drone\\26_05\\CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera.tif&rasterId=',
 '20826']

In [133]:
cursor = arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name','URL','ProductName'],q)
df = pd.DataFrame(cursor,columns=['Name','URL','ProductName'])

In [134]:
df = df[df.URL.notnull()].reset_index(drop=True)
df.head()

,Name,URL,ProductName
0,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera,https://sig.aminerals.cl/imgdyn/rest/services/...,
1,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,https://sig.aminerals.cl/imgdyn/rest/services/...,
2,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,https://sig.aminerals.cl/imgdyn/rest/services/...,
3,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125,https://sig.aminerals.cl/imgdyn/rest/services/...,
4,CL_MLP_PAO_IF_Ortho_26_05_10_ED1,https://sig.aminerals.cl/imgdyn/rest/services/...,


In [131]:
df.loc[2,'URL']

'https://sig.aminerals.cl/imgdyn/rest/services/CL_MLP_PAO/CL_MLP_PAO_IF_Ortho_Geosupport/ImageServer/file?id=.\\El_Mauro_Drone\\26_05\\CL_MLP_PAO_IF_Ortho_26_05_14_Camino_Alternativo_Salamanca.tif&rasterId='

In [ ]:
'https://sig.aminerals.cl/imgdyn/rest/services/CL_MLP_PAO/CL_MLP_PAO_IF_Ortho_Geosupport/ImageServer/file?id=.\\El_Mauro_Drone\\26_05\\CL_MLP_PAO_IF_Ortho_26_05_14_Camino_Alternativo_Salamanca.tif&rasterId='

In [144]:
import os
FOLDER_APRX = rf'{Path.cwd()}\APRX'
os.makedirs(FOLDER_APRX,exist_ok=True)

In [145]:
from pathlib import Path
import csv
import re
import arcpy


LOAD_RESULTS_CSV = Path.cwd() / "outputs" / "carga_mosaico" / "20260615_162625" / "01_load_results.csv"

aprx_path = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
map_name = "CL MLP PAO 27 Imagenes Aereas PAO Image Server"
PARENT_GROUP_NAME = "Vuelos Drone PAO"
TARGET_GROUP_NAME = "Imagenes Drone"
prefix = "CL_MLP_PAO_IF_Ortho_"
suffix = ".tif"
SAVE_COPY_FOR_REVIEW = True
ADD_ONLY_MISSING_TO_GROUP = True
APRX_COPY_PATH = fr"{FOLDER_APRX}\{Path(aprx_path).stem}_verificacion_carga_imagenes.aprx"


def layer_long_name(layer):
    try:
        return layer.longName
    except Exception:
        return layer.name


def short_layer_name(name):
    short_name = Path(str(name)).name

    while short_name.startswith("tmp_"):
        short_name = short_name.replace("tmp_", "", 1)

    if short_name.startswith(prefix):
        short_name = short_name.replace(prefix, "", 1)

    if short_name.endswith(suffix):
        short_name = short_name[:-len(suffix)]

    return short_name


def layer_name_from_path(path_value):
    return short_layer_name(Path(str(path_value)).name)


def comparable_layer_keys(name):
    short_name = short_layer_name(name)
    full_name = f"{prefix}{short_name}{suffix}"
    tmp_full_name = f"tmp_{full_name}"
    return {short_name.lower(), full_name.lower(), tmp_full_name.lower()}


def is_image_layer_name(name):
    short_name = short_layer_name(name)
    return bool(re.match(r"^\d{2}_\d{2}_\d{2}_.+", short_name))


def sort_key_newest_first(layer_name):
    name = short_layer_name(layer_name)
    match = re.search(r"^(\d{2})_(\d{2})_(\d{2})_", name)
    if match:
        yy, mm, dd = match.groups()
        return (int(yy), int(mm), int(dd), name.lower())
    return (0, 0, 0, name.lower())


def read_loaded_image_paths(csv_path):
    with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = csv.DictReader(file)
        return [
            row["destination_path"]
            for row in rows
            if row.get("overall_status") == "ok" and row.get("destination_path")
        ]


def is_descendant_of_group(layer, group_layer):
    return layer_long_name(layer).startswith(layer_long_name(group_layer) + "\\")


def is_direct_child_of_group(layer, group_layer):
    long_name = layer_long_name(layer)
    group_long_name = layer_long_name(group_layer)
    prefix_name = group_long_name + "\\"
    if not long_name.startswith(prefix_name):
        return False
    relative_name = long_name[len(prefix_name):]
    return "\\" not in relative_name


def direct_child_groups(map_obj, parent_group):
    return [
        lyr
        for lyr in map_obj.listLayers()
        if lyr.isGroupLayer and lyr != parent_group and is_direct_child_of_group(lyr, parent_group)
    ]


def group_direct_raster_layers(map_obj, group_layer):
    return [
        lyr
        for lyr in map_obj.listLayers()
        if lyr != group_layer and is_direct_child_of_group(lyr, group_layer) and lyr.isRasterLayer and not lyr.isGroupLayer
    ]


def find_group(map_obj, group_name):
    matches = [lyr for lyr in map_obj.listLayers() if lyr.isGroupLayer and lyr.name == group_name]
    if not matches:
        available = [layer_long_name(lyr) for lyr in map_obj.listLayers() if lyr.isGroupLayer]
        raise ValueError(f"No se encontro el grupo '{group_name}'. Grupos disponibles: {available}")
    if len(matches) > 1:
        print(f"Advertencia: se encontraron {len(matches)} grupos '{group_name}'. Se usara: {layer_long_name(matches[0])}")
    return matches[0]


def find_or_restore_target_group(map_obj, parent_group):
    expected_long_name = f"{layer_long_name(parent_group)}\\{TARGET_GROUP_NAME}"
    exact_matches = [
        lyr
        for lyr in direct_child_groups(map_obj, parent_group)
        if lyr.name == TARGET_GROUP_NAME or layer_long_name(lyr) == expected_long_name
    ]
    if exact_matches:
        return exact_matches[0]

    candidates = []
    for group in direct_child_groups(map_obj, parent_group):
        raster_children = group_direct_raster_layers(map_obj, group)
        image_children = [lyr for lyr in raster_children if is_image_layer_name(lyr.name)]
        if image_children:
            candidates.append((len(image_children), group))

    if candidates:
        candidates.sort(key=lambda item: item[0], reverse=True)
        target_group = candidates[0][1]
        original_group_name = target_group.name
        target_group.name = TARGET_GROUP_NAME
        print(f"Grupo destino restaurado: {original_group_name} -> {TARGET_GROUP_NAME}")
        return target_group

    available = [layer_long_name(lyr) for lyr in direct_child_groups(map_obj, parent_group)]
    raise ValueError(
        f"No se encontro el subgrupo '{TARGET_GROUP_NAME}' bajo '{PARENT_GROUP_NAME}' "
        f"ni un grupo candidato con imagenes. Subgrupos disponibles: {available}"
    )


def add_raster_to_target_group(map_obj, group_layer, image_path, target_name):
    before_layer_names = {layer_long_name(lyr) for lyr in group_direct_raster_layers(map_obj, group_layer)}
    safe_tmp_stem = re.sub(r"[^A-Za-z0-9_]+", "_", Path(image_path).stem)
    tmp_name = f"tmp_{safe_tmp_stem}"
    tmp_layer_file_path = Path.cwd() / f"{tmp_name}.lyrx"

    arcpy.management.MakeRasterLayer(image_path, tmp_name)
    arcpy.management.SaveToLayerFile(tmp_name, str(tmp_layer_file_path), "ABSOLUTE")
    tmp_layer_file = arcpy.mp.LayerFile(str(tmp_layer_file_path))

    try:
        result = map_obj.addLayerToGroup(group_layer, tmp_layer_file, "TOP")
        added_layer = None

        if isinstance(result, list) and result:
            added_layer = result[0]
        elif result is not None:
            added_layer = result

        if added_layer is None:
            after_layers = group_direct_raster_layers(map_obj, group_layer)
            new_layers = [lyr for lyr in after_layers if layer_long_name(lyr) not in before_layer_names]
            if new_layers:
                added_layer = new_layers[0]

        if added_layer is None:
            target_keys = comparable_layer_keys(target_name)
            for lyr in group_direct_raster_layers(map_obj, group_layer):
                if target_keys.intersection(comparable_layer_keys(lyr.name)):
                    added_layer = lyr
                    break

        if added_layer is not None:
            added_layer.name = target_name

        return added_layer
    finally:
        try:
            arcpy.management.Delete(tmp_name)
        except Exception:
            pass
        try:
            tmp_layer_file_path.unlink(missing_ok=True)
        except Exception:
            pass


def move_layer_to_top_of_group(map_obj, group_layer, layer_to_move):
    current_layers = group_direct_raster_layers(map_obj, group_layer)
    if not current_layers or layer_to_move == current_layers[0]:
        return
    map_obj.moveLayer(current_layers[0], layer_to_move, "BEFORE")


def order_group_layers_newest_first(map_obj, group_layer):
    ordered_layers = sorted(
        group_direct_raster_layers(map_obj, group_layer),
        key=lambda layer: sort_key_newest_first(layer.name),
        reverse=True,
    )
    for layer in reversed(ordered_layers):
        move_layer_to_top_of_group(map_obj, group_layer, layer)


image_paths_to_add = read_loaded_image_paths(LOAD_RESULTS_CSV)
print(f"Imagenes cargadas a revisar en APRX: {len(image_paths_to_add)}")

aprx = arcpy.mp.ArcGISProject(aprx_path)

try:
    maps = aprx.listMaps(map_name)

    if not maps:
        print("No se encontro el mapa")
    else:
        m = maps[0]
        parent_group = find_group(m, PARENT_GROUP_NAME)
        target_group = find_or_restore_target_group(m, parent_group)
        print(f"Grupo padre: {layer_long_name(parent_group)}")
        print(f"Grupo destino: {layer_long_name(target_group)}")

        if not is_direct_child_of_group(target_group, parent_group):
            raise ValueError(f"Grupo destino inesperado: {layer_long_name(target_group)}")

        existing_layers_before = group_direct_raster_layers(m, target_group)
        existing_keys = set()
        for lyr in existing_layers_before:
            existing_keys.update(comparable_layer_keys(lyr.name))
        print(f"Raster directos en '{TARGET_GROUP_NAME}' antes de agregar: {len(existing_layers_before)}")

        added_count = 0
        skipped_count = 0
        renamed_count = 0
        error_count = 0

        # 1. Agregar solo las imagenes nuevas como raster directos dentro de Vuelos Drone PAO > Imagenes Drone.
        for image_path in image_paths_to_add:
            target_name = layer_name_from_path(image_path)
            target_keys = comparable_layer_keys(target_name)

            if ADD_ONLY_MISSING_TO_GROUP and target_keys.intersection(existing_keys):
                print(f"Ya existe en el grupo, se conserva y se omite: {target_name}")
                skipped_count += 1
                continue

            try:
                added_layer = add_raster_to_target_group(m, target_group, image_path, target_name)
                if added_layer is None:
                    print(f"Advertencia: no se pudo confirmar el objeto retornado, se continuara con validacion posterior: {target_name}")
                existing_keys.update(target_keys)
                added_count += 1
                print(f"Agregada al grupo: {target_name}")
            except Exception as exc:
                error_count += 1
                print(f"Error agregando {image_path}: {exc}")

        # 2. Renombrar raster directos al formato corto: YY_MM_DD_Sector, sin prefijo, sin tmp_ y sin .tif.
        for lyr in group_direct_raster_layers(m, target_group):
            try:
                original_name = lyr.name
                new_name = short_layer_name(original_name)
                if new_name != original_name:
                    print(f"Renombrando: {original_name} -> {new_name}")
                    lyr.name = new_name
                    renamed_count += 1
            except Exception as exc:
                error_count += 1
                print(f"Error con capa {lyr.name}: {exc}")

        # 3. Ordenar de mas nueva a mas vieja dentro de Imagenes Drone.
        order_group_layers_newest_first(m, target_group)

        layers_after = group_direct_raster_layers(m, target_group)
        print(f"Raster directos en '{TARGET_GROUP_NAME}' despues de agregar: {len(layers_after)}")
        print(f"Resumen APRX: nuevas_agregadas={added_count}, existentes_conservadas={skipped_count}, renombradas={renamed_count}, errores={error_count}")

        if SAVE_COPY_FOR_REVIEW:
            aprx.saveACopy(APRX_COPY_PATH)
            print(f"Copia de verificacion guardada: {APRX_COPY_PATH}")
        else:
            aprx.save()
            print("Proyecto original guardado")
finally:
    del aprx


Imagenes cargadas a revisar en APRX: 32
Grupo padre: Vuelos Drone PAO
Grupo destino: Vuelos Drone PAO\Imagenes Drone
Raster directos en 'Imagenes Drone' antes de agregar: 468
Agregada al grupo: 26_05_01_Estacion_Cabecera
Agregada al grupo: 26_05_06_Subestacion-El-Mauro-1
Agregada al grupo: 26_05_06_Subestacion-El-Mauro-2
Agregada al grupo: 26_05_06_TORRE_E85_A_E_125
Agregada al grupo: 26_05_10_ED1
Agregada al grupo: 26_05_10_Helipuerto
Agregada al grupo: 26_05_10_Patio-19B-y-Armado
Agregada al grupo: 26_05_10_DME9-PA12-IIFF8
Agregada al grupo: 26_05_07_TORRES_E48_A_E84_PV4
Agregada al grupo: 26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3
Agregada al grupo: 26_05_13_DME9-PA12-IIFF8
Agregada al grupo: 26_05_13_Subestacion-El-Mauro_A_E35
Agregada al grupo: 26_05_13_Subestacion-El-Mauro-1
Agregada al grupo: 26_05_13_EM2_S2
Agregada al grupo: 26_05_13_EBD-1
Agregada al grupo: 26_05_13_ED2
Agregada al grupo: 26_05_13_Subestacion-El-Mauro-2
Agregada al grupo: 26_05_13_EBD-2
Agregada al grupo: 26_05

In [150]:
from pathlib import PurePath
PurePath(image_paths_to_add[0])

PureWindowsPath('//amssclgis10.ams.gmams.cl/CL_MLP_PAO/Chacay_Drone/26_05/CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera.tif')

In [16]:
from pathlib import Path
from datetime import datetime
import csv
import json
import shutil
import arcpy

LOAD_RESULTS_CSV = Path.cwd() / "outputs" / "carga_mosaico" / "20260615_162625" / "01_load_results.csv"
PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

GEOMETRY_SOURCE = "FEATURE_CLASS_ARCPY"  # FEATURE_CLASS_ARCPY o MOSAIC_DATASET
PATH_FC_FOOTPRINTS = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
FOOTPRINT_NAME_FIELD = "Name"

NORMALIZED_OUTPUT_ROOT = Path.cwd() / "outputs" / "normalizacion_footprintV7" 
NORMALIZE_METHOD = "RASTERIO_MASK"
REPLACE_ORIGINALS = False
CREATE_ORIGINAL_BACKUP = True
BUILD_PYRAMIDS_AND_STATS = True

arcpy.env.overwriteOutput = True


def read_loaded_rows(csv_path):
    with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = csv.DictReader(file)
        return [row for row in rows if row.get("overall_status") == "ok" and row.get("destination_path")]


def quote_sql_text(value):
    return str(value).replace("'", "''")


def name_where_clause(dataset, field_name, name):
    field = arcpy.AddFieldDelimiters(dataset, field_name)
    return f"{field} = '{quote_sql_text(name)}'"


def resolve_geometry_source():
    source = str(GEOMETRY_SOURCE).strip()
    if source in {"FEATURE_CLASS_ARCPY", "FEATURE_CLASS_GEOPANDAS", "MOSAIC_DATASET"}:
        if source == "FEATURE_CLASS_GEOPANDAS":
            # El ambiente ArcGIS Pro puede tener pandas/numpy incompatibles. Usamos ArcPy para leer el FC.
            source = "FEATURE_CLASS_ARCPY"
        return source, PATH_FC_FOOTPRINTS

    if source and source not in {"FEATURE_CLASS_ARCPY", "FEATURE_CLASS_GEOPANDAS", "MOSAIC_DATASET"}:
        return "FEATURE_CLASS_ARCPY", source

    return source, PATH_FC_FOOTPRINTS


def raster_clip_output_path(source_path, output_root):
    source = Path(source_path)
    datastore_parent = source.parent.name
    output_dir = output_root / datastore_parent
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / source.name


def arcpy_geometry_to_geojson_geometry(geometry):
    if hasattr(geometry, "__geo_interface__"):
        return geometry.__geo_interface__
    converted = arcpy.AsShape(json.loads(geometry.JSON), True)
    if hasattr(converted, "__geo_interface__"):
        return converted.__geo_interface__
    raise RuntimeError("No se pudo convertir la geometria ArcPy a GeoJSON")


def load_footprints_index(feature_class, name_field):
    if not feature_class:
        raise ValueError("PATH_FC_FOOTPRINTS esta vacio. Define la ruta al feature class de footprints.")

    fields = [field.name for field in arcpy.ListFields(feature_class)]
    if name_field not in fields:
        raise ValueError(f"El campo '{name_field}' no existe en {feature_class}. Campos disponibles: {fields}")

    index = {}
    duplicates = set()
    with arcpy.da.SearchCursor(feature_class, [name_field, "SHAPE@"]) as cursor:
        for name, geometry in cursor:
            if not name or not geometry:
                continue
            key = str(name).lower()
            if key in index:
                duplicates.add(str(name))
            index.setdefault(key, []).append(geometry)

    if duplicates:
        print(f"Advertencia: hay nombres duplicados en footprints: {sorted(duplicates)[:10]}")
    if not index:
        raise RuntimeError(f"No se encontraron geometrías en {feature_class}")

    print(f"Footprints cargados con ArcPy: {sum(len(v) for v in index.values())} desde {feature_class}")
    return index


def footprint_geometry_from_index(footprints_index, raster_name, target_spatial_reference):
    matches = footprints_index.get(str(raster_name).lower(), [])
    if len(matches) != 1:
        raise RuntimeError(f"Footprint esperado 1 en feature class, encontrado {len(matches)}, Name={raster_name}")

    geometry = matches[0]
    if target_spatial_reference and geometry.spatialReference and geometry.spatialReference.factoryCode != target_spatial_reference.factoryCode:
        geometry = geometry.projectAs(target_spatial_reference)

    return arcpy_geometry_to_geojson_geometry(geometry)


def selected_footprint_geometry_from_mosaic(mosaic_dataset, raster_name, target_spatial_reference):
    layer_name = f"footprint_{abs(hash(raster_name))}"
    where_clause = name_where_clause(mosaic_dataset, "Name", raster_name)
    arcpy.management.MakeFeatureLayer(mosaic_dataset, layer_name, where_clause)
    try:
        count = int(arcpy.management.GetCount(layer_name).getOutput(0))
        if count != 1:
            raise RuntimeError(f"Footprint esperado 1, encontrado {count}, Name={raster_name}")

        with arcpy.da.SearchCursor(layer_name, ["SHAPE@"]) as cursor:
            geometry = next(cursor)[0]

        if target_spatial_reference and geometry.spatialReference.factoryCode != target_spatial_reference.factoryCode:
            geometry = geometry.projectAs(target_spatial_reference)

        return arcpy_geometry_to_geojson_geometry(geometry)
    finally:
        arcpy.management.Delete(layer_name)


def normalize_with_rasterio_mask(source_path, footprint_geometry, output_path):
    import rasterio
    from rasterio.mask import mask

    with rasterio.open(source_path) as src:
        masked_data, out_transform = mask(src, [footprint_geometry], crop=True, filled=False)
        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            height=masked_data.shape[1],
            width=masked_data.shape[2],
            transform=out_transform,
            compress="LZW",
            BIGTIFF="IF_SAFER",
        )

        if src.count == 3 and src.nodata is None:
            rgb = masked_data.filled(0)
            alpha = (~masked_data.mask.all(axis=0)).astype("uint8") * 255
            profile.update(count=4, dtype=rgb.dtype, nodata=None)
            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(rgb, indexes=[1, 2, 3])
                dst.write(alpha, indexes=4)
                dst.colorinterp = (
                    rasterio.enums.ColorInterp.red,
                    rasterio.enums.ColorInterp.green,
                    rasterio.enums.ColorInterp.blue,
                    rasterio.enums.ColorInterp.alpha,
                )
        else:
            nodata_value = src.nodata if src.nodata is not None else 0
            data = masked_data.filled(nodata_value)
            profile.update(nodata=nodata_value)
            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(data)
                if src.colorinterp:
                    dst.colorinterp = src.colorinterp


def normalize_raster_by_footprint(source_path, raster_name, output_path, footprints_index=None):
    raster_sr = arcpy.Describe(source_path).spatialReference

    if ACTIVE_GEOMETRY_SOURCE == "FEATURE_CLASS_ARCPY":
        footprint_geometry = footprint_geometry_from_index(footprints_index, raster_name, raster_sr)
    elif ACTIVE_GEOMETRY_SOURCE == "MOSAIC_DATASET":
        footprint_geometry = selected_footprint_geometry_from_mosaic(PATH_MOSAIC_DATASET, raster_name, raster_sr)
    else:
        raise ValueError(f"GEOMETRY_SOURCE no soportado: {GEOMETRY_SOURCE} / activo: {ACTIVE_GEOMETRY_SOURCE}")

    normalize_with_rasterio_mask(source_path, footprint_geometry, output_path)

    if BUILD_PYRAMIDS_AND_STATS:
        arcpy.management.BuildPyramidsandStatistics(str(output_path))


def replace_original_with_backup(original_path, normalized_path):
    original = Path(original_path)
    normalized = Path(normalized_path)
    backup = original.with_suffix(original.suffix + ".bak_original")

    if CREATE_ORIGINAL_BACKUP and not backup.exists():
        shutil.copy2(original, backup)

    shutil.copy2(normalized, original)
    if BUILD_PYRAMIDS_AND_STATS:
        arcpy.management.BuildPyramidsandStatistics(str(original))
    return backup


rows = read_loaded_rows(LOAD_RESULTS_CSV)
NORMALIZED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ACTIVE_GEOMETRY_SOURCE, ACTIVE_FOOTPRINTS_PATH = resolve_geometry_source()

footprints_index = None
if ACTIVE_GEOMETRY_SOURCE == "FEATURE_CLASS_ARCPY":
    footprints_index = load_footprints_index(ACTIVE_FOOTPRINTS_PATH, FOOTPRINT_NAME_FIELD)

print(f"Imagenes a normalizar por footprint: {len(rows)}")
print(f"Metodo: {NORMALIZE_METHOD}")
print(f"Fuente geometria: {ACTIVE_GEOMETRY_SOURCE}")
print(f"Feature footprints: {ACTIVE_FOOTPRINTS_PATH}")
print(f"Salida de revision: {NORMALIZED_OUTPUT_ROOT}")
print(f"Reemplazar originales: {REPLACE_ORIGINALS}")

results = []
for row in rows:
    raster_name = row["Name"]
    source_path = row["destination_path"]
    output_path = raster_clip_output_path(source_path, NORMALIZED_OUTPUT_ROOT)

    item = {
        "Name": raster_name,
        "source_path": source_path,
        "normalized_path": str(output_path),
        "method": NORMALIZE_METHOD,
        "geometry_source": ACTIVE_GEOMETRY_SOURCE,
        "footprints_path": ACTIVE_FOOTPRINTS_PATH,
        "replace_original": REPLACE_ORIGINALS,
        "status": "pending",
        "error": "",
        "backup_path": "",
    }

    try:
        if not Path(source_path).exists():
            raise FileNotFoundError(source_path)

        normalize_raster_by_footprint(source_path, raster_name, output_path, footprints_index)

        if REPLACE_ORIGINALS:
            backup = replace_original_with_backup(source_path, output_path)
            item["backup_path"] = str(backup)
            item["status"] = "normalized_and_replaced"
        else:
            item["status"] = "normalized_for_review"

        print(f"OK {raster_name} -> {output_path}")
    except Exception as exc:
        item["status"] = "error"
        item["error"] = str(exc)
        print(f"ERROR {raster_name}: {exc}")

    results.append(item)
    

summary_csv = NORMALIZED_OUTPUT_ROOT / "00_normalizacion_footprint_resultados.csv"
with summary_csv.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0].keys()) if results else ["Name", "status"])
    writer.writeheader()
    writer.writerows(results)

status_counts = {}
for item in results:
    status_counts[item["status"]] = status_counts.get(item["status"], 0) + 1

print("Resumen normalizacion:", status_counts)
print(f"CSV resultados: {summary_csv}")



Advertencia: hay nombres duplicados en footprints: ['CL_MLP_PAO_IF_Ortho_26_04_17_EB3', 'CL_MLP_PAO_IF_Ortho_26_04_18_Helipuerto', 'CL_MLP_PAO_IF_Ortho_26_04_22_EB3', 'CL_MLP_PAO_IF_Ortho_26_04_23_TORRES_E35_A_E48', 'CL_MLP_PAO_IF_Ortho_26_04_24_Patio-19B-y-Armado', 'CL_MLP_PAO_IF_Ortho_26_04_25_EBD', 'CL_MLP_PAO_IF_Ortho_26_04_25_Pozos_PRP']
Footprints cargados con ArcPy: 481 desde \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO
Imagenes a normalizar por footprint: 32
Metodo: RASTERIO_MASK
Fuente geometria: FEATURE_CLASS_ARCPY
Feature footprints: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO
Salida de revision: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\normalizacion_footprintV7
Reemplazar originales: False



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\envs\arcgispro\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\envs\arcgispro\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\envs\arcgispro\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\en

AttributeError: _ARRAY_API not found

OK CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera -> c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\normalizacion_footprintV7\26_05\CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera.tif



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\envs\arcgispro\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\envs\arcgispro\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\envs\arcgispro\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\en

...

AttributeError: _ARRAY_API not found

OK CL_MLP_PAO_IF_Ortho_26_05_14_TORRES_E31_A_E48_PV4-2 -> c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\normalizacion_footprintV7\26_05\CL_MLP_PAO_IF_Ortho_26_05_14_TORRES_E31_A_E48_PV4-2.tif



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\envs\arcgispro\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\envs\arcgispro\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\envs\arcgispro\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\esrlrivero_adm\AppData\Local\ESRI\conda\en

AttributeError: _ARRAY_API not found

OK CL_MLP_PAO_IF_Ortho_26_05_15_EM1 -> c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\normalizacion_footprintV7\26_05\CL_MLP_PAO_IF_Ortho_26_05_15_EM1.tif
Resumen normalizacion: {'normalized_for_review': 30, 'error': 2}
CSV resultados: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\normalizacion_footprintV7\00_normalizacion_footprint_resultados.csv
